# P1-30 · Labelling rule against the final mini-MIAS expert masks (CPU, ~2 min)
Run after P1-26b. Reads `mias_ext/masks_proxy` and `revision_checks/masks_final`; writes `revision_checks/rule_vs_final_per_image.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, numpy as np, pandas as pd, cv2
OUT = '/content/drive/MyDrive/P1_v2'; MIAS = f'{OUT}/mias_ext'; RC = f'{OUT}/revision_checks'
mm = pd.read_csv(MIAS + '/manifest.csv'); mm = mm[mm.status == 'ok'].sort_values('file_id')
def dice(a, b):
    s = a.sum() + b.sum(); return 1.0 if s == 0 else 2 * (a & b).sum() / s
rows = []
for f in mm.file_id:
    p = cv2.imread(f'{MIAS}/masks_proxy/{f}.png', 0) > 127
    rows.append(dict(file_id=f, rule_dice_full=dice(p, cv2.imread(f'{RC}/masks_final/full/{f}.png', 0) > 127),
                     rule_dice_breast=dice(p, cv2.imread(f'{RC}/masks_final/breast/{f}.png', 0) > 127)))
R = pd.DataFrame(rows); R.to_csv(f'{RC}/rule_vs_final_per_image.csv', index=False)
print(len(R), 'images |', R[['rule_dice_full', 'rule_dice_breast']].agg(['mean', 'std']).round(4).to_string())